## Weather Project

This project is an end to end machine learning (ML) application that predicts, tomorrow's temperature and weather, based on London's historical data, between 1979 and 2021.

The dataset contains over 15,000+ observations with features; cloud cloud_cover, sunshine, global radiation, temperature, precipitation, pressure, snow depth.

This project involves data cleaning, exploratory data analysis (EDA) using interactive visualizations, feature engeneering, model training (regression / classification) and evaluation. The final models will be integrated into a fully interactive Streamlit web application.

- By Mark

---

### 1. Loading Data

In [63]:
# Imports
import pandas as pd
import numpy as np
import plotly.express as px
from datetime import datetime

from sklearn.tree import DecisionTreeRegressor, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle 

In [64]:
# Load dataset
df = pd.read_csv('data/london_weather.csv')

# Structure
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns}")

Shape: (15341, 10)
Columns: Index(['date', 'cloud_cover', 'sunshine', 'global_radiation', 'max_temp',
       'mean_temp', 'min_temp', 'precipitation', 'pressure', 'snow_depth'],
      dtype='object')


### 2. Cleaning Data

In [65]:
# Missing values
df = df.dropna(how="all")
print(f"Missing values\n{df.isna().sum()}")

Missing values
date                   0
cloud_cover           19
sunshine               0
global_radiation      19
max_temp               6
mean_temp             36
min_temp               2
precipitation          6
pressure               4
snow_depth          1441
dtype: int64


In [66]:
# Fill missing values with 0
df['cloud_cover'] = df['cloud_cover'].fillna(0)
df['snow_depth'] = df['snow_depth'].fillna(0)
df['precipitation'] = df['precipitation'].fillna(0)

# drop values that cannot be filled in
df = df.dropna(subset=["min_temp", "max_temp", "global_radiation", "pressure"])
df["mean_temp"] = df["mean_temp"].fillna((df["max_temp"] + df["min_temp"]) / 2)

In [67]:
df["date"] = pd.to_datetime(df["date"], format="%Y%m%d")

In [68]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day_of_year

In [69]:
winter = [12,1,2]
spring = [3,4,5]
summer = [6,7,8]
autumn = [9,10,11]

def get_season(month):
    if month in winter:
        return "winter"
    elif month in spring:
        return "spring"
    elif month in autumn:
        return "autumn"
    else:
        return "summer"

df["season"] = df["month"].apply(get_season)

In [70]:
df.head()

,date,cloud_cover,sunshine,global_radiation,max_temp,mean_temp,min_temp,precipitation,pressure,snow_depth,year,month,day,season
0,1979-01-01,2.0,7.0,52.0,2.3,-4.1,-7.5,0.4,101900.0,9.0,1979,1,1,winter
1,1979-01-02,6.0,1.7,27.0,1.6,-2.6,-7.5,0.0,102530.0,8.0,1979,1,2,winter
2,1979-01-03,5.0,0.0,13.0,1.3,-2.8,-7.2,0.0,102050.0,4.0,1979,1,3,winter
3,1979-01-04,8.0,0.0,13.0,-0.3,-2.6,-6.5,0.0,100840.0,2.0,1979,1,4,winter
4,1979-01-05,6.0,2.0,29.0,5.6,-0.8,-1.4,0.0,102250.0,1.0,1979,1,5,winter


### 3. Exploratory Data Analysis (EDA)

In [71]:
fig1 = px.scatter(df, x="date", y="mean_temp", color="season",
           opacity=0.4, trendline_color_override="black",
           title="Mean Temperature Over Time (1979-2021)",
           labels={"mean_temp": "Mean Temerature (°C)", "date": "Date (year)"})

fig1.show()
fig1.write_html("figures/temp_over_time.html")

This plot shows the long term temprature trend in London from 1979 to 2021. We can see that there are month / season based fluctuations in temprature, with summer months being warmer and winter months being colder. However, there is also a slight upward trend in temprature over the years, indicating that London is getting warmer, likley due to climate change.

In [72]:
fig2 = px.scatter(df, x="day", y="mean_temp", color="season",
           opacity=0.4, animation_frame=("year"), title="Mean Daily Temperature by Year",
           labels={"mean_temp": "Mean Temerature (°C)", "date": "Date (year)"})

fig2.show()
fig2.write_html("figures/mean_daily_temp_by_year.html")

In [73]:
fig = px.line(df, x="day", y=["pressure"], animation_frame="year", title="Animated Daily Pressure Curve by Year",
              labels={"pressure": "Pressure (Pa)", "day": "Day of Year"})
fig.show()

In [74]:
fig3 = px.bar(df, x="year", y="snow_depth", title="Daily Snow Depth",
              labels={"snow_depth": "Snow Depth (mm)", "year": "Year"}, height=1000)

fig3.show()
fig3.write_html("figures/daily_snow_depth.html")

In [75]:
fig = px.line(df, x="day", y="snow_depth", animation_frame="year", title="Animated Daily Pressure Curve by Year",
              labels={"pressure": "Pressure (Pa)", "day": "Day of Year"}, height=400)
fig.show()

In [76]:
fig = px.line(df, x="day", y=["pressure", "snow_depth"], animation_frame="year", title="Animated Daily Pressure Curve by Year",
              labels={"pressure": "Pressure (Pa)", "day": "Day of Year"})
fig.show()

In [77]:
fig4 = px.line(df, x="day", y="precipitation", animation_frame="year", title="Precipitation Over Time",
              labels={"precipitation": "Precipitation (mm)", "year": "Year"})
fig4.show()
fig4.write_html("figures/mean_daily_temp_by_year.html")




In [78]:
fig_3d = px.scatter_3d(
    df,
    x="day",
    y="mean_temp",
    z="pressure",
    color="season",
    opacity=1,
    title="Mean Temperature & Pressure by Year (3D)",
    labels={
        "day": "Day of Year",
        "mean_temp": "Mean Temperature (°C)",
        "pressure": "Pressure (Pa)"
    }, height=1000
)
fig_3d.update_traces(marker=dict(size=2))
fig_3d.show()
fig_3d.write_html("figures/precipitation_snow_3d.html", include_plotlyjs="cdn")

***

### 4. Feature Engineering

We need to teach the model what kinds of patterns matter, just like a human weather forcaster learns

To do this we create new feathures that help the model understand:
* What happened yesterday
* What has been happening over the last few days
* What season it is
* Weather the recent weather patterns are stable or changing

In [79]:
df.head()

,date,cloud_cover,sunshine,global_radiation,max_temp,mean_temp,min_temp,precipitation,pressure,snow_depth,year,month,day,season
0,1979-01-01,2.0,7.0,52.0,2.3,-4.1,-7.5,0.4,101900.0,9.0,1979,1,1,winter
1,1979-01-02,6.0,1.7,27.0,1.6,-2.6,-7.5,0.0,102530.0,8.0,1979,1,2,winter
2,1979-01-03,5.0,0.0,13.0,1.3,-2.8,-7.2,0.0,102050.0,4.0,1979,1,3,winter
3,1979-01-04,8.0,0.0,13.0,-0.3,-2.6,-6.5,0.0,100840.0,2.0,1979,1,4,winter
4,1979-01-05,6.0,2.0,29.0,5.6,-0.8,-1.4,0.0,102250.0,1.0,1979,1,5,winter


In [80]:
df["temp_tomorrow"] = df["mean_temp"].shift(-1)

In [81]:
# 1 -> It will rain, 0 _> It won't rain
df["tomorrow_rain"] = (df["precipitation"].shift(-1) > 0).astype(int)

In [82]:
df.head()

,date,cloud_cover,sunshine,global_radiation,max_temp,mean_temp,min_temp,precipitation,pressure,snow_depth,year,month,day,season,temp_tomorrow,tomorrow_rain
0,1979-01-01,2.0,7.0,52.0,2.3,-4.1,-7.5,0.4,101900.0,9.0,1979,1,1,winter,-2.6,0
1,1979-01-02,6.0,1.7,27.0,1.6,-2.6,-7.5,0.0,102530.0,8.0,1979,1,2,winter,-2.8,0
2,1979-01-03,5.0,0.0,13.0,1.3,-2.8,-7.2,0.0,102050.0,4.0,1979,1,3,winter,-2.6,0
3,1979-01-04,8.0,0.0,13.0,-0.3,-2.6,-6.5,0.0,100840.0,2.0,1979,1,4,winter,-0.8,0
4,1979-01-05,6.0,2.0,29.0,5.6,-0.8,-1.4,0.0,102250.0,1.0,1979,1,5,winter,-0.5,1


In [83]:
df = df_season = pd.get_dummies(df, columns=["season"], drop_first=True)

In [84]:
df.head()

,date,cloud_cover,sunshine,global_radiation,max_temp,mean_temp,min_temp,precipitation,pressure,snow_depth,year,month,day,temp_tomorrow,tomorrow_rain,season_spring,season_summer,season_winter
0,1979-01-01,2.0,7.0,52.0,2.3,-4.1,-7.5,0.4,101900.0,9.0,1979,1,1,-2.6,0,False,False,True
1,1979-01-02,6.0,1.7,27.0,1.6,-2.6,-7.5,0.0,102530.0,8.0,1979,1,2,-2.8,0,False,False,True
2,1979-01-03,5.0,0.0,13.0,1.3,-2.8,-7.2,0.0,102050.0,4.0,1979,1,3,-2.6,0,False,False,True
3,1979-01-04,8.0,0.0,13.0,-0.3,-2.6,-6.5,0.0,100840.0,2.0,1979,1,4,-0.8,0,False,False,True
4,1979-01-05,6.0,2.0,29.0,5.6,-0.8,-1.4,0.0,102250.0,1.0,1979,1,5,-0.5,1,False,False,True


In [85]:
df["temp_yesterday"] = df["mean_temp"].shift(1)
df["rain_yesterday"] = df["precipitation"].shift(1)
df["sunshine_yesterday"] = df["sunshine"].shift(1)
df["pressure_yesterday"] = df["pressure"].shift(1)
df["cloud_yesterday"] = df["cloud_cover"].shift(1)

In [86]:
df.head()

,date,cloud_cover,sunshine,global_radiation,max_temp,mean_temp,min_temp,precipitation,pressure,snow_depth,...,temp_tomorrow,tomorrow_rain,season_spring,season_summer,season_winter,temp_yesterday,rain_yesterday,sunshine_yesterday,pressure_yesterday,cloud_yesterday
0,1979-01-01,2.0,7.0,52.0,2.3,-4.1,-7.5,0.4,101900.0,9.0,...,-2.6,0,False,False,True,NaN,NaN,NaN,NaN,NaN
1,1979-01-02,6.0,1.7,27.0,1.6,-2.6,-7.5,0.0,102530.0,8.0,...,-2.8,0,False,False,True,-4.1,0.4,7.0,101900.0,2.0
2,1979-01-03,5.0,0.0,13.0,1.3,-2.8,-7.2,0.0,102050.0,4.0,...,-2.6,0,False,False,True,-2.6,0.0,1.7,102530.0,6.0
3,1979-01-04,8.0,0.0,13.0,-0.3,-2.6,-6.5,0.0,100840.0,2.0,...,-0.8,0,False,False,True,-2.8,0.0,0.0,102050.0,5.0
4,1979-01-05,6.0,2.0,29.0,5.6,-0.8,-1.4,0.0,102250.0,1.0,...,-0.5,1,False,False,True,-2.6,0.0,0.0,100840.0,8.0


In [87]:
df.head()

,date,cloud_cover,sunshine,global_radiation,max_temp,mean_temp,min_temp,precipitation,pressure,snow_depth,...,temp_tomorrow,tomorrow_rain,season_spring,season_summer,season_winter,temp_yesterday,rain_yesterday,sunshine_yesterday,pressure_yesterday,cloud_yesterday
0,1979-01-01,2.0,7.0,52.0,2.3,-4.1,-7.5,0.4,101900.0,9.0,...,-2.6,0,False,False,True,NaN,NaN,NaN,NaN,NaN
1,1979-01-02,6.0,1.7,27.0,1.6,-2.6,-7.5,0.0,102530.0,8.0,...,-2.8,0,False,False,True,-4.1,0.4,7.0,101900.0,2.0
2,1979-01-03,5.0,0.0,13.0,1.3,-2.8,-7.2,0.0,102050.0,4.0,...,-2.6,0,False,False,True,-2.6,0.0,1.7,102530.0,6.0
3,1979-01-04,8.0,0.0,13.0,-0.3,-2.6,-6.5,0.0,100840.0,2.0,...,-0.8,0,False,False,True,-2.8,0.0,0.0,102050.0,5.0
4,1979-01-05,6.0,2.0,29.0,5.6,-0.8,-1.4,0.0,102250.0,1.0,...,-0.5,1,False,False,True,-2.6,0.0,0.0,100840.0,8.0


In [88]:
df["mean_temp"].rolling(3).mean()

0             NaN
1             NaN
2       -3.166667
3       -2.666667
4       -2.066667
           ...   
15336    5.000000
15337    4.500000
15338    3.733333
15339    2.133333
15340    1.500000
Name: mean_temp, Length: 15311, dtype: float64

In [89]:
df["temp_yesterday"] = df["mean_temp"].shift(1)
df["rain_yesterday"] = df["precipitation"].shift(1)
df["sunshine_yesterday"] = df["sunshine"].shift(1)
df["pressure_yesterday"] = df["pressure"].shift(1)
df["cloud_yesterday"] = df["cloud_cover"].shift(1)

df["temp_last3"] = df["mean_temp"].rolling(3).mean()
df["pressure_last3"] = df["pressure"].rolling(3).mean()
df["sun_last3"] = df["sunshine"].rolling(3).mean()
df["cloud_last3"] = df["cloud_cover"].rolling(3).mean()
df["rain_last3"] = df["precipitation"].rolling(3).mean()

df["temp_last5"] = df["mean_temp"].rolling(5).mean()
df["pressure_last5"] = df["pressure"].rolling(5).mean()
df["sun_last5"] = df["sunshine"].rolling(5).mean()
df["cloud_last5"] = df["cloud_cover"].rolling(5).mean()
df["rain_last5"] = df["precipitation"].rolling(5).mean()

In [90]:
df = df.iloc[4:-1]

In [91]:
df.to_csv("data/features.csv", index=False)
print(f"Saved {len(df)} rows to data/features.csv")

Saved 15306 rows to data/features.csv


In [92]:
df.head()

,date,cloud_cover,sunshine,global_radiation,max_temp,mean_temp,min_temp,precipitation,pressure,snow_depth,...,temp_last3,pressure_last3,sun_last3,cloud_last3,rain_last3,temp_last5,pressure_last5,sun_last5,cloud_last5,rain_last5
4,1979-01-05,6.0,2.0,29.0,5.6,-0.8,-1.4,0.0,102250.0,1.0,...,-2.066667,101713.333333,0.666667,6.333333,0.000000,-2.58,101914.0,2.14,5.4,0.08
5,1979-01-06,5.0,3.8,39.0,8.3,-0.5,-6.6,0.7,102780.0,1.0,...,-1.300000,101956.666667,1.933333,6.333333,0.233333,-1.86,102090.0,1.50,6.0,0.14
6,1979-01-07,8.0,0.0,13.0,8.5,1.5,-5.3,5.2,102520.0,0.0,...,0.066667,102516.666667,1.933333,6.333333,1.966667,-1.04,102088.0,1.16,6.4,1.18
7,1979-01-08,8.0,0.1,15.0,5.8,6.9,5.3,0.8,101870.0,0.0,...,2.633333,102390.000000,1.300000,7.000000,2.233333,0.90,102052.0,1.18,7.0,1.34
8,1979-01-09,4.0,5.8,50.0,5.2,3.7,1.6,7.2,101170.0,0.0,...,4.033333,101853.333333,1.966667,6.666667,4.400000,2.16,102118.0,2.34,6.2,2.78


### 5. Decision Tree for Feature Importance

Terms:
* Node: The box with the question
* Leaf: The final box with a prediction
* Root: The first question
* Split: The act of dividing the data based on a question
* Feature: The question asked in each node

In [93]:
df.columns

Index(['date', 'cloud_cover', 'sunshine', 'global_radiation', 'max_temp',
       'mean_temp', 'min_temp', 'precipitation', 'pressure', 'snow_depth',
       'year', 'month', 'day', 'temp_tomorrow', 'tomorrow_rain',
       'season_spring', 'season_summer', 'season_winter', 'temp_yesterday',
       'rain_yesterday', 'sunshine_yesterday', 'pressure_yesterday',
       'cloud_yesterday', 'temp_last3', 'pressure_last3', 'sun_last3',
       'cloud_last3', 'rain_last3', 'temp_last5', 'pressure_last5',
       'sun_last5', 'cloud_last5', 'rain_last5'],
      dtype='object')

In [94]:
df = df.drop(columns=["date", "tomorrow_rain", "day", "month"])

In [95]:
df.head()

,cloud_cover,sunshine,global_radiation,max_temp,mean_temp,min_temp,precipitation,pressure,snow_depth,year,...,temp_last3,pressure_last3,sun_last3,cloud_last3,rain_last3,temp_last5,pressure_last5,sun_last5,cloud_last5,rain_last5
4,6.0,2.0,29.0,5.6,-0.8,-1.4,0.0,102250.0,1.0,1979,...,-2.066667,101713.333333,0.666667,6.333333,0.000000,-2.58,101914.0,2.14,5.4,0.08
5,5.0,3.8,39.0,8.3,-0.5,-6.6,0.7,102780.0,1.0,1979,...,-1.300000,101956.666667,1.933333,6.333333,0.233333,-1.86,102090.0,1.50,6.0,0.14
6,8.0,0.0,13.0,8.5,1.5,-5.3,5.2,102520.0,0.0,1979,...,0.066667,102516.666667,1.933333,6.333333,1.966667,-1.04,102088.0,1.16,6.4,1.18
7,8.0,0.1,15.0,5.8,6.9,5.3,0.8,101870.0,0.0,1979,...,2.633333,102390.000000,1.300000,7.000000,2.233333,0.90,102052.0,1.18,7.0,1.34
8,4.0,5.8,50.0,5.2,3.7,1.6,7.2,101170.0,0.0,1979,...,4.033333,101853.333333,1.966667,6.666667,4.400000,2.16,102118.0,2.34,6.2,2.78


In [96]:
# 1. Splitting the data in

# Feature
X = df.drop(columns=["temp_tomorrow"])
 
# Targets
y = df["temp_tomorrow"]

# Splitting
train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [97]:
X.shape, y.shape

((15306, 28), (15306,))

In [98]:
X_train.shape, X_test.shape

((12244, 28), (3062, 28))

In [99]:
tree = DecisionTreeRegressor(random_state=42)
tree.fit(X_train, y_train)

,criterion,'squared_error'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [100]:
# 2. Prediction
y_test_pred = tree.predict(X_test)

In [101]:
y_test_pred

array([12.2, 19. ,  6.8, ..., 16.6,  2. , 17.1], shape=(3062,))

In [102]:
# 3. Model Fitting
df_test_plot = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_test_pred
})



df_test_plot.head()

,Actual,Predicted
0,13.7,12.2
1,18.0,19.0
2,7.1,6.8
3,16.6,16.0
4,9.6,9.6


In [103]:
features = X.columns
importance = pd.Series(tree.feature_importances_, index=X.columns).sort_values(ascending=False)
importance = importance.reset_index(name="importance")

In [104]:
importance.T

,0,1,2,3,4,5,6,7,8,9,...,18,19,20,21,22,23,24,25,26,27
index,max_temp,mean_temp,cloud_cover,min_temp,global_radiation,sunshine,temp_last5,year,pressure,temp_yesterday,...,pressure_last3,pressure_yesterday,rain_last3,cloud_yesterday,rain_yesterday,cloud_last3,season_spring,season_winter,season_summer,snow_depth
importance,0.931183,0.027877,0.004474,0.003459,0.002923,0.002645,0.002264,0.002163,0.002038,0.001849,...,0.001255,0.001212,0.000989,0.000934,0.000878,0.000683,0.000455,0.000184,0.000096,0.000038


In [105]:
fig_importance = px.bar(importance, x="index", y="importance", title="Feature Importance", log_y=True)
fig_importance.show()

fig_importance.write_html("figures/importance.html")

In [106]:
df

,cloud_cover,sunshine,global_radiation,max_temp,mean_temp,min_temp,precipitation,pressure,snow_depth,year,...,temp_last3,pressure_last3,sun_last3,cloud_last3,rain_last3,temp_last5,pressure_last5,sun_last5,cloud_last5,rain_last5
4,6.0,2.0,29.0,5.6,-0.8,-1.4,0.0,102250.0,1.0,1979,...,-2.066667,101713.333333,0.666667,6.333333,0.000000,-2.58,101914.0,2.14,5.4,0.08
5,5.0,3.8,39.0,8.3,-0.5,-6.6,0.7,102780.0,1.0,1979,...,-1.300000,101956.666667,1.933333,6.333333,0.233333,-1.86,102090.0,1.50,6.0,0.14
6,8.0,0.0,13.0,8.5,1.5,-5.3,5.2,102520.0,0.0,1979,...,0.066667,102516.666667,1.933333,6.333333,1.966667,-1.04,102088.0,1.16,6.4,1.18
7,8.0,0.1,15.0,5.8,6.9,5.3,0.8,101870.0,0.0,1979,...,2.633333,102390.000000,1.300000,7.000000,2.233333,0.90,102052.0,1.18,7.0,1.34
8,4.0,5.8,50.0,5.2,3.7,1.6,7.2,101170.0,0.0,1979,...,4.033333,101853.333333,1.966667,6.666667,4.400000,2.16,102118.0,2.34,6.2,2.78
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15335,0.0,2.1,38.0,10.0,4.9,-0.1,12.0,101960.0,0.0,2020,...,4.533333,102296.666667,2.300000,2.666667,4.000000,7.46,101682.0,1.38,4.6,2.88
15336,1.0,0.9,32.0,7.5,7.5,7.6,2.0,98000.0,0.0,2020,...,5.000000,101020.000000,2.166667,2.333333,4.666667,6.62,101088.0,1.56,3.2,3.16
15337,7.0,3.7,38.0,3.6,1.1,-1.3,0.2,97370.0,0.0,2020,...,4.500000,99110.000000,2.233333,2.666667,4.733333,4.44,100452.0,2.30,3.2,2.84
15338,7.0,0.0,21.0,4.1,2.6,1.1,0.0,98830.0,0.0,2020,...,3.733333,98066.666667,1.533333,5.000000,0.733333,3.74,99852.0,2.04,4.2,2.84


In [107]:
# Up to 0.002038 importance
final_features = ["max_temp", "mean_temp", "cloud_cover", "min_temp", "temp_last5", "year", "pressure"]

In [108]:
final_features

['max_temp',
 'mean_temp',
 'cloud_cover',
 'min_temp',
 'temp_last5',
 'year',
 'pressure']

In [109]:
X = df[final_features]

In [110]:
X

,max_temp,mean_temp,cloud_cover,min_temp,temp_last5,year,pressure
4,5.6,-0.8,6.0,-1.4,-2.58,1979,102250.0
5,8.3,-0.5,5.0,-6.6,-1.86,1979,102780.0
6,8.5,1.5,8.0,-5.3,-1.04,1979,102520.0
7,5.8,6.9,8.0,5.3,0.90,1979,101870.0
8,5.2,3.7,4.0,1.6,2.16,1979,101170.0
...,...,...,...,...,...,...,...
15335,10.0,4.9,0.0,-0.1,7.46,2020,101960.0
15336,7.5,7.5,1.0,7.6,6.62,2020,98000.0
15337,3.6,1.1,7.0,-1.3,4.44,2020,97370.0
15338,4.1,2.6,7.0,1.1,3.74,2020,98830.0


#### Regression Models and Evaluation
We'll start with a simple linear regression model as a baseline.

In [111]:
from sklearn.linear_model import LinearRegression

In [112]:
X = df.drop(columns=["temp_tomorrow"])
X = X[final_features]
y = df["temp_tomorrow"]

train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [113]:
model_reg = LinearRegression()
model_reg.fit(X_train, y_train)

y_pred = model_reg.predict(X_test)

In [114]:

mae = mean_absolute_error(y_test, y_pred) # (real, predicted)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2:", r2)

MAE: 0.8526989972271333
RMSE: 1.1039288215909033
R2: 0.9615072339328307


* MAE: On average the model's temprature predictions are off by less 1C.
* RMSE: The models typical error is around 1C meaning the model rarely makes large mistakes.
* R2 The model explains about 93% of the variance of tomorrows temperature based on today's weather.

In [115]:
df_eval = pd.DataFrame({
    "Actual": y_test,
    "Predicted": y_pred
})

In [116]:
scatter = px.scatter(df_eval, x="Actual", y="Predicted", title="Actual vs Predicted Temprature (Test Set)", trendline="ols")
scatter.show()

scatter.write_html("figures/scatter.html")

### 6. Saving and Exporting Models

In [117]:
# Saving the model
with open("models/temperature_model.pkl", "wb") as f:
    pickle.dump(model_reg, f)
    
with open("models/temperature_features.pkl", "wb") as f:
    pickle.dump(final_features, f)

In [118]:
# Loading the model
with open("models/temperature_model.pkl", "rb") as f:
    model = pickle.load(f)
with open("models/temperature_features.pkl", "rb") as f:
    feature_list = pickle.load(f)

In [119]:
print(feature_list)

['max_temp', 'mean_temp', 'cloud_cover', 'min_temp', 'temp_last5', 'year', 'pressure']


In [120]:
# Try out our model
user_data = {
 "max_temp": 9.0,        # Max temp yesterday in °C
 "min_temp": 5.0,        # Min temp yesterday in °C
 "cloud_cover": 60,      # Approx % of cloud cover yesterday (predominantly broken/partly cloudy)  
 "pressure": 1000 * 100, # 1000 mbar in Pascals = 100000 Pa  
 "year": 2026,           # Current year  
 "last1": 7.0,           # 1 day before yesterday mean temp in °C 
 "last2": 6.8,           # 2 days before mean temp
 "last3": 7.2,           # 3 days before mean temp 
 "last4": 7.4,           # 4 days before mean temp
 "last5": 7.1            # 5 days before mean temp
}

In [121]:
# Mean temperature (°C)
mean_temp = (user_data["max_temp"] + user_data["min_temp"]) / 2
# 5-day rolling mean temperature (°C)
temp_last5 = np.mean([
    user_data["last1"],
    user_data["last2"],
    user_data["last3"],
    user_data["last4"],
    user_data["last5"]
])
# Pressure: mbar → Pa
pressure_pa = user_data["pressure"] * 100
# Cloud: % -> Okat
cloud_oktas = round(user_data["cloud_cover"] / 100 * 8)
input_row = {
    "max_temp": user_data["max_temp"],
    "mean_temp": mean_temp,
    "cloud_cover": cloud_oktas,
    "min_temp": user_data["min_temp"],
    "temp_last5": temp_last5,
    "year": user_data["year"],
    "pressure": pressure_pa
}

In [122]:
X_input = pd.DataFrame([input_row])

In [123]:
X_input

,max_temp,mean_temp,cloud_cover,min_temp,temp_last5,year,pressure
0,9.0,7.0,5,5.0,7.1,2026,10000000


In [124]:
prediction = model.predict(X_input)[0]
print(f"Predicted temperature tomorrow: {prediction:.2f} °C")

Predicted temperature tomorrow: 5.05 °C
